# Scraping database review

This notebook browses `scraping.db`, the scraping module's only implemented database. The top-level `src/storage/` module is still a skeleton, so it has no schema or data to review.

> **Kernel note:** select the project's `.venv` Python interpreter/kernel in VS Code so `pandas` and `src.scraping` imports resolve.

In [10]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Find the repository regardless of the notebook launch directory."""
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'config.yaml').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DB_PATH = REPO_ROOT / 'scraping.db'
print(f'Repository: {REPO_ROOT}')
print(f'Database:   {DB_PATH} ({DB_PATH.stat().st_size if DB_PATH.exists() else 0:,} bytes)')

Repository: /Users/kumo/programming/competitor_product_search
Database:   /Users/kumo/programming/competitor_product_search/scraping.db (4,096 bytes)


In [11]:
from src.scraping.storage import ScrapeDB

db = ScrapeDB(DB_PATH)
db.init_db()  # Idempotent: creates the six tables on a fresh, empty database.

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    db.conn,
)
table_summary = pd.DataFrame(
    [
        {'table': name, 'rows': db.conn.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0]}
        for name in tables['name']
    ]
)
display(table_summary)

,table,rows
0,escalations,0
1,golden_samples,0
2,invalid_target_phrases,0
3,parsers,0
4,results,0
5,scrape_runs,0


## `scrape_runs`

The operational log: URL, scraper path, outcome, latency, cost, and the parser used when applicable.

In [12]:
runs = pd.read_sql_query('SELECT * FROM scrape_runs ORDER BY id DESC', db.conn)
display(runs)

runs_with_parser = pd.read_sql_query(
    """
    SELECT r.*, p.site AS parser_site, p.version AS parser_version
    FROM scrape_runs AS r
    LEFT JOIN parsers AS p ON p.id = r.winning_parser_id
    ORDER BY r.id DESC
    """,
    db.conn,
)
display(runs_with_parser)

,id,url,host,site,scraper,scraped_at,outcome,path,winning_parser_id,attempts,model_used,latency_ms,cost


,id,url,host,site,scraper,scraped_at,outcome,path,winning_parser_id,attempts,model_used,latency_ms,cost,parser_site,parser_version


## `results`

This is the main scraped-product data table. The overview keeps JSON compact; the second view expands `product_data` into its individual ProductData fields.

In [ ]:
def shorten(value: object, limit: int = 180) -> object:
    if value is None or pd.isna(value):
        return value
    text = str(value)
    return text if len(text) <= limit else text[:limit] + ' …'


def decode_json(value: object) -> dict:
    if not value:
        return {}
    try:
        decoded = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return {'_unparseable_product_data': value}
    return decoded if isinstance(decoded, dict) else {'_json_value': decoded}


results = pd.read_sql_query('SELECT * FROM results ORDER BY id DESC', db.conn)
results_overview = results.copy()
if 'product_data' in results_overview:
    results_overview['product_data'] = results_overview['product_data'].map(shorten)
display(results_overview)

result_metadata = results.reindex(columns=['id', 'url', 'site', 'scraped_at'])
result_fields = pd.json_normalize(results['product_data'].map(decode_json).tolist())
results_flat = result_metadata.join(result_fields)
display(results_flat)

,id,url,site,scraped_at,product_data


,id,url,site,scraped_at


## `parsers`

Generated parser code is intentionally shortened here. Use `show_blob` below to inspect a full parser.

In [5]:
parsers = pd.read_sql_query('SELECT * FROM parsers ORDER BY id DESC', db.conn)
parsers_overview = parsers.copy()
if 'code' in parsers_overview:
    parsers_overview['code'] = parsers_overview['code'].map(shorten)
display(parsers_overview)

,id,site,version,code,page_type_scope,status,created_at,created_by


## `golden_samples`

Golden snapshots validate newly repaired parsers. The flattened view makes expected ProductData fields easy to compare.

In [6]:
goldens = pd.read_sql_query('SELECT * FROM golden_samples ORDER BY id DESC', db.conn)
goldens_overview = goldens.copy()
for column in ['html_snapshot', 'expected_output']:
    if column in goldens_overview:
        goldens_overview[column] = goldens_overview[column].map(shorten)
display(goldens_overview)

golden_metadata = goldens.reindex(columns=['id', 'site', 'page_type', 'captured_at', 'is_stale'])
golden_fields = pd.json_normalize(goldens['expected_output'].map(decode_json).tolist())
goldens_flat = golden_metadata.join(golden_fields)
display(goldens_flat)

,id,site,page_type,html_snapshot,expected_output,captured_at,is_stale


,id,site,page_type,captured_at,is_stale


## `escalations`

Escalations are deduplicated by signature. The second view uses the app's store API for the currently open queue.

In [7]:
from src.scraping.storage import EscalationStore

escalations = pd.read_sql_query('SELECT * FROM escalations ORDER BY id DESC', db.conn)
escalations_overview = escalations.copy()
if 'snapshot' in escalations_overview:
    escalations_overview['snapshot'] = escalations_overview['snapshot'].map(shorten)
display(escalations_overview)

open_escalations = pd.DataFrame(EscalationStore(db).get_open())
display(open_escalations)

,id,signature,reason,affected_count,snapshot,status,created_at


""


## `invalid_target_phrases`

Small lookup table used to recognize pages that are not product targets.

In [8]:
phrases = pd.read_sql_query('SELECT * FROM invalid_target_phrases ORDER BY id DESC', db.conn)
display(phrases)

,id,site,phrase,source,added_at


## Full-field drill-down

Overview tables truncate long code and snapshots. Call `show_blob` with a table, row id, and column to print the complete value; JSON is formatted for readability.

In [9]:
REVIEW_TABLES = {
    'parsers', 'golden_samples', 'scrape_runs', 'results', 'escalations', 'invalid_target_phrases'
}


def show_blob(table: str, row_id: int, column: str) -> None:
    """Print one complete text/JSON field from a reviewed table."""
    if table not in REVIEW_TABLES:
        raise ValueError(f'Unknown review table: {table}')
    valid_columns = {row['name'] for row in db.conn.execute(f'PRAGMA table_info({table})')}
    if column not in valid_columns:
        raise ValueError(f'Unknown column for {table}: {column}')

    row = db.conn.execute(
        f'SELECT {column} FROM {table} WHERE id = ?', (row_id,)
    ).fetchone()
    if row is None:
        raise LookupError(f'No {table} row with id={row_id}')

    value = row[0]
    if value is None:
        print('(NULL)')
        return
    try:
        print(json.dumps(json.loads(value), indent=2, ensure_ascii=False, default=str))
    except (TypeError, json.JSONDecodeError):
        print(value)


# Examples:
# show_blob('parsers', 1, 'code')
# show_blob('golden_samples', 1, 'html_snapshot')
# show_blob('results', 1, 'product_data')
# show_blob('escalations', 1, 'snapshot')

## Handy filtered questions

For common site-level questions, the purpose-built stores are more convenient than writing the aggregation again. Change `SITE` and re-run this cell.

In [ ]:
from src.scraping.storage import ParserStore, RunStore

SITE = 'tesco'
active_parsers = pd.DataFrame(ParserStore(db).get_active_ordered_by_hits(SITE))
parser_hit_rates = pd.DataFrame(RunStore(db).get_hit_rates(SITE))

print(f'Active parsers for {SITE}:')
display(active_parsers)
print(f'Parser hit rates for {SITE}:')
display(parser_hit_rates)